# RegDrift: train and evaluate the LoRA clause extractor
**Runtime > Change runtime type > T4 GPU** before you start. Free Colab is enough.

Flow: Groq proposes labels for your circulars -> **you review them** -> build train/test (test = whole circulars the model never sees) -> LoRA fine-tune a 0.5B model -> compare against the keyword baseline and Groq.

Colab's disk is wiped when the session ends, so run the *Save* cell after reviewing.

In [ ]:
REPO_URL = "https://github.com/Ajitesh1234/RegDrift.git"  # change if your repo name differs
!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt -r requirements-train.txt

In [ ]:
import os
from google.colab import userdata
for k in ["GROQ_API_KEY", "JIRA_URL", "JIRA_EMAIL", "JIRA_TOKEN", "JIRA_PROJECT"]:
    try: os.environ[k] = userdata.get(k)
    except Exception: print("not set (optional):", k)

## (Optional) restore earlier work from Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/regdrift
!cp -n /content/drive/MyDrive/regdrift/candidates.jsonl data/ 2>/dev/null; ls data

## 1. Groq proposes labels (resumable; re-run if rate-limited)

In [ ]:
!python distill.py --delay 3

## 2. You review them: a = accept, n = no rule, e = edit, s = skip, q = save and quit
This is the step that makes the dataset trustworthy. Read each clause; do not just press `a`.

In [ ]:
%run review_candidates.py

## Save your reviews (do this after every session)

In [ ]:
!cp data/candidates.jsonl /content/drive/MyDrive/regdrift/
!python review_candidates.py --stats

## 3. Build train / test files
Needs 4+ reviewed circulars for a held-out test set.

In [ ]:
!python build_lora_data.py
!cat data/split_report.json

## 4. Train (about 5-15 minutes on a T4)

In [ ]:
!python train_lora.py --epochs 6

## 5. Evaluate on circulars the model never saw

In [ ]:
import os
os.environ["LORA_ADAPTER"] = "regdrift-lora"
!python evaluate.py --backends rules,groq,lora

## 6. Download your results
Put `regdrift-lora/` and `docs/eval_results.md` in your repo and quote the table honestly in your README.

In [ ]:
!zip -qr regdrift-lora.zip regdrift-lora
from google.colab import files
files.download("regdrift-lora.zip")
files.download("docs/eval_results.md")
files.download("data/candidates.jsonl")

## Use the tuned model in the pipeline
```
LORA_ADAPTER=regdrift-lora python regdrift.py circulars/some.pdf --dry-run
```
Extraction then runs clause by clause on your adapter and falls back to Groq or the keyword rules if it fails.